# Module 6: Store Agent Memory as a Graph in Neo4j

**Purpose:** Test the storage and access rules behind agent memory. This notebook does not add memory retrieval to the booking agent.

This notebook stores one hotel preference in Neo4j. It reads that preference for the same actor in a new session and returns no preference for a second actor. It also traces the record to its source message and `Hotel` node. The notebook writes long-term preference memory and two short-term messages. It writes no reasoning traces.

**Overview**

- **Preference:** A stored statement about what an actor wants.
- **Provenance:** Links from a preference to its source message, session, and `Hotel` node.
- **Actor scope:** A query starts at one `User` node and returns only that actor's memories.
- **Entity linking:** The source message points to the canonical `Hotel` it mentions.
- **Preference promotion:** A confirmed preference becomes durable memory in a separate write.
- **Embedding:** Titan Text Embeddings V2 creates a vector when the notebook saves a preference.

Two controls protect memory access:

- **Multi-tenant writes:** Every memory write requires a user identifier. The application must still authenticate each actor and authorize access to session IDs.
- **Actor-scoped reads:** In version 0.5.0, the library's semantic searches cover the entire store. This notebook starts each read query at the selected `User`.

**Prerequisites:** The hotel graph must contain exactly one `Hotel` named `AnyCompany Cairo Nile View`. Configure Neo4j credentials for the correct database and AWS credentials that can invoke Titan Text Embeddings V2 in `AWS_REGION`. This module uses the same Neo4j instance and credentials as the earlier modules. The repository `.env` described in the top-level README contains those settings. `load_config` reads this folder's `.env` first, then the repository `.env`, then the repository `CONFIG.txt`. Values already set in the environment win. Live cells skip when credentials are unavailable.

In [ ]:
# On the Vocareum path, dependencies are pre-installed. Run this cell as-is.
# Running it anywhere else: uncomment the line below first.
# !pip install "neo4j-agent-memory[bedrock]==0.5.0" boto3 python-dotenv

print("Environment ready")

## 1. Create an isolated set of IDs for this workshop run

Every actor and session identifier includes a short run ID. This separates the current run from earlier transcripts. All identifiers also use the `memory06-` prefix. `cleanup_memory.py` uses this shared namespace to remove every workshop run. Each live cell opens and closes its own memory client. A failed cell cannot leave a connection open for the next cell.

In [ ]:
import os
import sys
from pathlib import Path

# The shared workshop/ package lives in notebooks/. These lines find that
# directory and put it on the import path. Everything else this notebook
# needs to start is in workshop/bootstrap.py.
_here = Path.cwd().resolve()
_named = os.environ.get("WORKSHOP_NOTEBOOKS_DIR") or _here
_starts = (Path(_named).expanduser().resolve(), _here, _here / "notebooks")
for _candidate in (*_starts, *_here.parents):
    if (_candidate / "workshop" / "bootstrap.py").is_file():
        sys.path.insert(0, str(_candidate))
        break

from workshop.bootstrap import start_module

NOTEBOOKS_ROOT, REPO_ROOT, MODULE_DIR = start_module("06-neo4j-memory")
print(f"Workshop root: {REPO_ROOT}")

In [ ]:
import uuid
from contextlib import asynccontextmanager

import boto3

from memory_helpers import (
    DEMO_ID_PREFIX,
    HERO_HOTEL_NAME,
    WORKSHOP_OWNER,
    build_memory_client,
    get_actor_preferences_for_hotel,
    link_message_to_hotel,
    link_preference_to_message_and_hotel,
    load_config,
    tag_demo_records,
)

RUN_ID = uuid.uuid4().hex[:8]
ACTOR_A = f"{DEMO_ID_PREFIX}{RUN_ID}-guest-alice"
ACTOR_B = f"{DEMO_ID_PREFIX}{RUN_ID}-guest-blake"
SESSION_A1 = f"{DEMO_ID_PREFIX}{RUN_ID}-session-a1"
SESSION_A2 = f"{DEMO_ID_PREFIX}{RUN_ID}-session-a2"
SESSION_B1 = f"{DEMO_ID_PREFIX}{RUN_ID}-session-b1"

try:
    config = load_config()
except RuntimeError as exc:
    config = None
    print(f"Neo4j is not configured: {exc}")

MEMORY_READY = (
    config is not None and boto3.Session().get_credentials() is not None
)

@asynccontextmanager
async def open_memory():
    memory = build_memory_client(config)
    await memory.connect()
    try:
        yield memory
    finally:
        await memory.close()

if MEMORY_READY:
    print(f"Run {RUN_ID}: Neo4j at {config.uri}, Bedrock in {config.region}.")
else:
    print("Not configured. Every live cell below will skip.")

**Run the notebook from top to bottom once per session.** To start another run, execute the `RUN_ID` cell again. Reusing the same run ID creates duplicate message records.

## 2. Store a fixed hotel preference scenario

The notebook stores fixed message content so every run uses the same preference scenario. Each run still creates new identifiers and records. This keeps the exercise focused on Neo4j. The first query confirms the `Hotel` node exists and raises a clear error when it is missing.

The source turn names a hotel the application has already resolved. After storing the message, the workshop writes `(:Message)-[:MENTIONS]->(:Hotel)` to the canonical domain node. In a free-form conversation, an extractor can propose the entities first. When an agent tool already returned a stable hotel identifier, linking that known subject directly is faster and more reliable.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        hotels = await memory.query.cypher(
            """
            CYPHER 25
            MATCH (h:Hotel {name: $hotel_name})
            RETURN h.name AS name
            """,
            {"hotel_name": HERO_HOTEL_NAME},
        )
        if len(hotels) != 1:
            raise RuntimeError(
                f"The hotel graph must contain exactly one Hotel named "
                f"{HERO_HOTEL_NAME!r}; found {len(hotels)}."
            )

        preference_source = await memory.short_term.add_message(
            SESSION_A1,
            "user",
            f"I loved staying at {HERO_HOTEL_NAME}. A room on a high "
            "floor away from the elevator is a must for me.",
            user_identifier=ACTOR_A,
            extraction_mode="skip",
        )
        await memory.short_term.add_message(
            SESSION_A1,
            "assistant",
            "I will remember that hotel and room preference.",
            user_identifier=ACTOR_A,
            extraction_mode="skip",
        )

    mentioned = link_message_to_hotel(
        config, str(preference_source.id), HERO_HOTEL_NAME
    )
    assert mentioned, "source message or unique hero Hotel was missing"
    print(f"Stored two fixture messages in {SESSION_A1}.")
    print("Linked the source message to the canonical Hotel.")

## 3. Promote the confirmed preference to durable memory

Entity linking answers what a conversation is about. A separate memory policy decides what is safe and useful to retain. In this fixed scenario, the user's statement is the confirmed preference candidate. The library creates a `Preference` node owned by the actor, and the workshop adds two provenance relationships:

- **`DERIVED_FROM`:** It links the preference to its exact source message.
- **`ABOUT_HOTEL`:** It links the preference to the same canonical `Hotel` the message mentions.

The full pattern is `(u:User)-[:HAS_PREFERENCE]->(p:Preference)-[:DERIVED_FROM]->(m:Message)-[:MENTIONS]->(h:Hotel)` and `(p)-[:ABOUT_HOTEL]->(h)`. The workshop preserves the `Hotel` node. It does not add an `Entity` label or memory properties.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        preference = await memory.long_term.add_preference(
            category=f"hotels-{DEMO_ID_PREFIX}{RUN_ID}",
            preference=(
                f"Loves {HERO_HOTEL_NAME} and wants a room on a high "
                "floor away from the elevator."
            ),
            context=f"Workshop run {RUN_ID}, session {SESSION_A1}",
            user_identifier=ACTOR_A,
        )

    linked = link_preference_to_message_and_hotel(
        config,
        str(preference.id),
        str(preference_source.id),
        HERO_HOTEL_NAME,
    )
    assert linked, "preference, message, or unique hero Hotel was missing"
    print("Preference linked to its source message and the real Hotel.")

## 4. Read the preference in a new session and check a second actor

**Purpose:** Confirm that a saved preference remains available to the same actor and stays hidden from another actor.

Actor A starts `SESSION_A2`. The application reads Actor A's saved preference for the named hotel with an actor-scoped Cypher query. Actor B starts a separate session. The same query returns no preference for Actor B. In production, the application must bind actor and session IDs to authenticated callers.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        await memory.short_term.add_message(
            SESSION_A2,
            "user",
            "What hotel and room preference do you have for me?",
            user_identifier=ACTOR_A,
            extraction_mode="skip",
        )
        await memory.short_term.add_message(
            SESSION_B1,
            "user",
            "What hotel and room preference do you have for me?",
            user_identifier=ACTOR_B,
            extraction_mode="skip",
        )

    for_a = get_actor_preferences_for_hotel(
        config, ACTOR_A, HERO_HOTEL_NAME
    )
    for_b = get_actor_preferences_for_hotel(
        config, ACTOR_B, HERO_HOTEL_NAME
    )
    assert len(for_a) == 1, f"actor A expected one preference, got {len(for_a)}"
    assert not for_b, f"actor B unexpectedly saw {len(for_b)} preference(s)"
    print(f"Actor A in fresh session: {for_a[0]['preference']}")
    print("Actor B: no preference returned.")

## 5. Trace the preference back to its source and hotel

Run one parameterized Cypher query. It returns the actor, preference, source message, source session, and canonical `Hotel`.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        rows = await memory.query.cypher(
            """
            CYPHER 25
            MATCH (u:User {identifier: $actor})
                  -[:HAS_PREFERENCE]->(p:Preference)
                  -[:DERIVED_FROM]->(m:Message)-[:MENTIONS]->(h:Hotel),
                  (m)<-[:HAS_MESSAGE]-(c:Conversation),
                  (p)-[:ABOUT_HOTEL]->(h {name: $hotel_name})
            RETURN u.identifier AS actor,
                   p.preference AS preference,
                   m.content AS source_message,
                   c.session_id AS source_session,
                   h.name AS hotel
            """,
            {"actor": ACTOR_A, "hotel_name": HERO_HOTEL_NAME},
        )
    assert len(rows) == 1, f"expected one provenance path, got {len(rows)}"
    for key, value in rows[0].items():
        print(f"{key:15s} {value}")

## Run marker for cleanup

The IDs use the `memory06-` namespace. An ownership marker gives cleanup a second way to identify workshop data. Cleanup removes namespaced memory records, tagged orphaned preferences, and workshop-owned relationships. It preserves every `Hotel` node.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    marked = tag_demo_records(
        config,
        session_ids=[SESSION_A1, SESSION_A2, SESSION_B1],
        user_identifiers=[ACTOR_A, ACTOR_B],
    )
    print(f"Marked {marked} record(s) with {WORKSHOP_OWNER!r}.")

## Conceptual comparison: managed memory and graph memory

**Purpose:** Compare AgentCore Memory with the Neo4j graph memory used in this notebook.

| Dimension | AgentCore Memory | Neo4j graph memory in Module 6 |
|-----------|------------------|----------------------------------|
| Entity and preference extraction | Managed asynchronous extraction | Application-controlled extraction, canonical linking, and promotion policy |
| Long-term availability | Extracted memories become available after background processing | Confirmed preferences become available when the write transaction commits |
| Inspectability | Retrieved through a service API | Queryable graph with source provenance |
| Correction path | Managed through the Memory service API | Append a replacement and supersede the old preference |
| Domain linking | Separate from domain data | `MENTIONS` and `ABOUT_HOTEL` point to the real `Hotel` |
| Isolation | Actor namespaces managed by the service | Scoped writes, actor-anchored reads, and application-side session authorization |
| Operations | AWS operates the store | You operate Neo4j and the embedding contract |

**Choose AgentCore Memory:** Use it when managed extraction and managed operations fit the application.

**Choose Neo4j graph memory:** Use it when you need application-controlled promotion, immediate recall, source provenance, and relationships to domain data.

A production system can use both. The important design boundary is not "extraction or no extraction." Entity extraction identifies what a turn is about; memory policy decides what becomes durable.

## Remove this notebook's memory records when you finish

The final cell can remove every run in the `memory06-` namespace while preserving `Hotel` nodes. Cleanup is disabled by default. **Run All** leaves the completed exercise available for inspection. Set `CLEAN_UP_DEMO_MEMORY` to `True` when you finish.

In [ ]:
from cleanup_memory import run_cleanup

CLEAN_UP_DEMO_MEMORY = False

if config is None:
    print("Cleanup skipped: Neo4j is not configured.")
elif not CLEAN_UP_DEMO_MEMORY:
    print("Cleanup skipped. Set CLEAN_UP_DEMO_MEMORY = True when finished.")
else:
    cleanup_result = run_cleanup(config)
    assert cleanup_result == 0, f"cleanup returned {cleanup_result}"
